In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *

In [39]:
def kernel(window = 3, dim = 2, exterior_total_weight = 1):
	K = np.ones([window] * dim)
	K = K * exterior_total_weight / (np.sum(K) - 1)
	K[tuple([slice(1, -1)] * dim)] = 1
	return K
	
def local_weighted_average(tensor, K = None):
	if K is None: K = kernel(dim = tensor.ndim - 1)
	assert tensor.ndim == K.ndim + 1, "Tensor and kernel must matching number of dimensions"
	M = convolve(tensor.astype(float).sum(axis=-1), K, mode='constant', cval=0.0)
	print(M)
	M = M / convolve(np.ones(tensor.shape[:-1]), K, mode='constant', cval=0.0)
	return M

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	print(f"Simulating for rho={rho}, pi={pi}, model={model.__name__}")
	simulate_in_grid(4, 5, model, (rho, pi), transformations=TRANSFORMS)

Simulating for rho=0.25, pi=0.1, model=betaWSBM
Progress: 4/64 (6.2%). Elapsed: 00:00:05. ETA: 00:01:25.
Progress: 8/64 (12.5%). Elapsed: 00:00:10. ETA: 00:01:14.
Progress: 12/64 (18.8%). Elapsed: 00:00:15. ETA: 00:01:07.
Progress: 16/64 (25.0%). Elapsed: 00:00:20. ETA: 00:01:01.
Progress: 20/64 (31.2%). Elapsed: 00:00:24. ETA: 00:00:53.
Progress: 24/64 (37.5%). Elapsed: 00:00:28. ETA: 00:00:47.
Progress: 28/64 (43.8%). Elapsed: 00:00:34. ETA: 00:00:43.
Progress: 32/64 (50.0%). Elapsed: 00:00:39. ETA: 00:00:39.
Progress: 36/64 (56.2%). Elapsed: 00:00:43. ETA: 00:00:33.
Progress: 40/64 (62.5%). Elapsed: 00:00:47. ETA: 00:00:28.
Progress: 44/64 (68.8%). Elapsed: 00:00:52. ETA: 00:00:23.
Progress: 48/64 (75.0%). Elapsed: 00:00:58. ETA: 00:00:19.
Progress: 52/64 (81.2%). Elapsed: 00:01:01. ETA: 00:00:14.
Progress: 56/64 (87.5%). Elapsed: 00:01:05. ETA: 00:00:09.
Progress: 60/64 (93.8%). Elapsed: 00:01:11. ETA: 00:00:04.
Progress: 64/64 (100.0%). Elapsed: 00:01:17. ETA: 00:00:00.
Simulating

KeyboardInterrupt: 

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	for p in linspace_exclusive(0, 1, 4):
		print(f"Simulating for rho={rho}, pi={pi}, model={model.__name__}, p11={p}")
		simulate_in_line(200, 20, model, (rho, pi), transformations=TRANSFORMS, p11 = p)

Simulating for rho=0.25, pi=0.1, model=betaWSBM, p11=0.2
Progress: 1/40 (2.5%). Elapsed: 00:00:01. ETA: 00:01:10.
Progress: 2/40 (5.0%). Elapsed: 00:00:02. ETA: 00:00:50.
Progress: 3/40 (7.5%). Elapsed: 00:00:03. ETA: 00:00:43.
Progress: 4/40 (10.0%). Elapsed: 00:00:04. ETA: 00:00:40.
Progress: 5/40 (12.5%). Elapsed: 00:00:05. ETA: 00:00:37.
Progress: 6/40 (15.0%). Elapsed: 00:00:06. ETA: 00:00:36.
Progress: 7/40 (17.5%). Elapsed: 00:00:07. ETA: 00:00:34.
Progress: 8/40 (20.0%). Elapsed: 00:00:08. ETA: 00:00:34.
Progress: 9/40 (22.5%). Elapsed: 00:00:09. ETA: 00:00:33.
Progress: 10/40 (25.0%). Elapsed: 00:00:11. ETA: 00:00:33.
Progress: 11/40 (27.5%). Elapsed: 00:00:12. ETA: 00:00:33.
Progress: 12/40 (30.0%). Elapsed: 00:00:13. ETA: 00:00:32.
Progress: 13/40 (32.5%). Elapsed: 00:00:15. ETA: 00:00:31.
Progress: 14/40 (35.0%). Elapsed: 00:00:16. ETA: 00:00:30.
Progress: 15/40 (37.5%). Elapsed: 00:00:17. ETA: 00:00:29.
Progress: 16/40 (40.0%). Elapsed: 00:00:18. ETA: 00:00:28.
Progress: 1

KeyboardInterrupt: 

In [ ]:
metrics = {}
for rho, pi in product(RHOS, PIS):
	metrics[(rho, pi)] = {}
	for model, model_params in MODELS_AND_PARAMS:
		m = model(rho, pi, model_params)
		A, Z = m(42)
		metrics[(rho, pi)][m] = {}
		for t in TRANSFORMS:
			metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z)

In [ ]:
for rho, pi in product(RHOS, PIS):
	plot_embedding(rho, pi, metrics[(rho, pi)])

In [ ]:
path = "Computation/No eigen-scaling/Grids"
metrics_g = {}
for rho, pi, model in RHOS_PIS_MODELS:
	grids = np.load(f"{path}/{model.__name__}_{r_dot(rho)}_{r_dot(pi)}.npz")
	metrics_g[(rho, pi, model)] = {}
	for t in TRANSFORMS:
		metrics_g[(rho, pi, model)][t] = {}
		for metric in METRICS_ID:
			metrics_g[(rho, pi, model)][t][metric] = grids[f'{t.id}_{metric}']

metrics_g = aggregate_metrics(metrics_g)

metrics_g = best_transform_metrics(metrics_g)

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	metrics_g[(rho, pi, model)] = best_transform_metrics(m)
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		metrics_g[(rho, pi, model)][t] = correlation(m)
		metrics_g[(rho, pi, model)][t] = bias(m)

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Pro

In [ ]:
#plot_scatter_Rand_vs_Chernoff(metrics_g, n_points_ratio_displayed=0.5)
art_plot_scatter_Rand_vs_Chernoff(metrics_g, n_points_ratio_displayed=0.5)

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)
		#art_plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)

In [ ]:
# Prendre moins de place première ligne

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plot_bias_heatmap(rho, pi, model, t, m, log = True)
		#art_plot_bias_heatmap(rho, pi, model, t, m, log = True)

In [ ]:
# Rand moyen, Regret moyen pour Best transform overall
# Puis découpage en 2 régions (Regret > 0 et Regret = 0) et Rand moyen et Regret moyen

# Average(Area C-estim-Best Transform) over 8 graphs (TreeMap)
# Pour chaque Transform élue Best Transform
#	  - Rand moyen + Regret moyen
#     Puis découpage en 2 régions (Regret > 0 et Regret = 0) et Rand moyen et Regret moyen

# Average(Area Rand-Best Transform) over 8 graphs (TreeMap)
#     Rand moyen pour chaque Best transform

# Average(Rand) over 8 graphs for 4 transforms + Best

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	plot_best_transform_heatmaps(rho, pi, model, m)
	#art_plot_best_transform_heatmaps(rho, pi, model, m)

In [ ]:
path = "Computation/No eigen-scaling/Lines"
metrics_l = {}
for rho, pi, model in RHOS_PIS_MODELS:
	grids = np.load(f"{path}/{model.__name__}_{r_dot(rho)}_{r_dot(pi)}.npz")
	metrics_l[(rho, pi, model)] = {}
	for tid, t in TRANSFORMS_MAP.items():
		metrics_l[(rho, pi, model)][t] = {}
		for metric in METRICS_ID:
			metrics_l[(rho, pi, model)][t][metric] = grids[f'{tid}_{metric}']
			
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_l[(rho, pi, model)]
	metrics_l[(rho, pi, model)] = best_transform_metrics(m)

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
def chernoffs_lines(rho, pi, model, metrics_l, metrics_g, param, n=n, transform_i = 0):
		chernoffs = ['C_graph', 'C_embed']
		
		fig, axes = plt.subplots(2, 1, figsize=(10, 12))
		fig.suptitle(f"Chernoffs of Transforms on Model: {model.name}\n" + model_str(n, rho, pi), fontsize=14)
		
		for ax, C in zip(axes, chernoffs):
			
			for t in [TRANSFORMS[transform_i]]:
				y_l = metrics_l[t][C]
				y_g = metrics_g[t][C][80, :]
				ax.plot(linspace_exclusive(0, 1, len(y_l)), y_l,
						label=f'{t.name} line',
						color=TRANSFORMS_CMAP[t],
						linewidth=2)
				ax.plot(linspace_exclusive(0, 1, len(y_g)), y_g,
						label=f'{t.name} grid',
						color=TRANSFORMS_CMAP[t],
						linewidth=2, linestyle='dashed')
			
			title = METRICS_MAP[C] + " of Transforms"
			ax.set_title(title, fontsize=12)
			#ax.set_ylim(-0.1, 1.05)
			ax.set_xlabel(f'{model.param_name}{sub(" " + param)}', fontsize=12)
			ax.set_ylabel(METRICS_ID_COSMETIC_MAP[C], fontsize=12)
			ax.set_xticks(np.linspace(0, 1, 5))
			ax.legend(loc="upper left", handlelength=2, handleheight=2, fontsize=9)
		
		plt.tight_layout()
		save_file('Plots/Best_Transform', f'{transform_i}_Chernoffs_Line_vs_Grid_{model.name}_{rho}_{pi}', dpi=300)

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	m_l = metrics_l[(rho, pi, model)]
	m_g = metrics_g[(rho, pi, model)]
	for i in range(4):
		chernoffs_lines(rho, pi, model, m_l, m_g, '12', transform_i = i)